# 01 — Data Cleaning

This notebook loads the raw AHI data from Excel, fixes header rows, and exports each sheet as a clean CSV for downstream analysis.

**Data source:** [Shimberg Center Assisted Housing Inventory](http://flhousingdata.shimberg.ufl.edu/assisted-housing-inventory/results?nid=100) — Alachua County, FL.

**Missing data conventions:** When a value is missing in a numeric column, we use `NaN` so the column stays numeric and missing values are automatically excluded from calculations. For text or categorical columns, we use `N/A`. Columns that use `x` and `-` to indicate whether something applies are converted to `True` and `False`.

In [7]:
import pandas as pd

file_path = "../Data/Raw Data/AHI_Raw_Data.xlsx"

df_funder = pd.read_excel(file_path, sheet_name="Sheet 1")
df_inventory = pd.read_excel(file_path, sheet_name="Sheet 2")
df_assistance_program = pd.read_excel(file_path, sheet_name="Sheet 3")
df_lost_property = pd.read_excel(file_path, sheet_name="Sheet 4")
df_housing_agencies = pd.read_excel(file_path, sheet_name="Sheet 5")

print("df_funder:", df_funder.shape)
print("df_inventory:", df_inventory.shape)
print("df_assistance_program:", df_assistance_program.shape)
print("df_lost_property:", df_lost_property.shape)
print("df_housing_agencies:", df_housing_agencies.shape)

df_funder: (8, 7)
df_inventory: (60, 168)
df_assistance_program: (10, 38)
df_lost_property: (11, 27)
df_housing_agencies: (4, 13)


### Header Fix

All 5 sheets contain a title row followed by a blank row before the actual column headers. We re-read each sheet using `header=2` to skip those two rows and set the third row as the proper header.

In [8]:
df_funder = pd.read_excel(file_path, sheet_name="Sheet 1", header=2)
df_inventory = pd.read_excel(file_path, sheet_name="Sheet 2", header=2)
df_assistance_program = pd.read_excel(file_path, sheet_name="Sheet 3", header=2)
df_lost_property = pd.read_excel(file_path, sheet_name="Sheet 4", header=2)
df_housing_agencies = pd.read_excel(file_path, sheet_name="Sheet 5", header=2)

print("df_funder:", df_funder.shape)
print("df_inventory:", df_inventory.shape)
print("df_assistance_program:", df_assistance_program.shape)
print("df_lost_property:", df_lost_property.shape)
print("df_housing_agencies:", df_housing_agencies.shape)

df_funder: (6, 7)
df_inventory: (58, 168)
df_assistance_program: (8, 38)
df_lost_property: (9, 27)
df_housing_agencies: (2, 13)


### df_inventory ###
In the original spreadsheet, '-' represents no and 'x' represents yes. '-' was replaced with 0 and 'x' was replaced with 1 for data consistency. 0 and 1 are used to represent data in boolean form, which suits how a funding source was either present or not present for a property.

In [9]:
#Cleaned Data
df_inventory = df_inventory.replace({'-': 0, 'x': 1, 'not avail.': 0})
df_inventory.head()

C:\Users\yimol\AppData\Local\Temp\ipykernel_7160\2518823016.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_inventory = df_inventory.replace({'-': 0, 'x': 1, 'not avail.': 0})


,Shim ID,FHFC Key,HUD REMS,Public Housing Development #,Florida DOR Parcel,Development Name,Street Address,City,Zip Code,County,...,"% of Households with at least one child age 0-17, 2018-2022, Tract","% of Households with at least one child age 0-17, 2018-2022, County","% of Households with at least one person age 65 or older, 2018-2022, Tract","% of Households with at least one person age 65 or older, 2018-2022, County",Total Living Area,Number of Buildings,REAC Score (HUD only),REAC Inspection Date,Construction Type (FHFC Only)',In unincorporated area
0,1213,0,800003897,0,3926002000,Alachua Apts,13605 NW County Road 235,Alachua,32615,Alachua,...,19.46,22.34,32.83,26.18,56979.0,15.0,96,2025-05-20,0,NaN
1,112,0,0,0,3563000000,Alachua Villas,14000 NW 154TH AVE,Alachua,32615,Alachua,...,34.30,22.34,38.12,26.18,26660.0,9.0,0,0,0,NaN
2,7872,3308,0,0,3214035000,Arbours at Merrillwood I,13207 NW 153rd Place,Alachua,32615,Alachua,...,39.30,22.34,25.72,26.18,NaN,NaN,0,0,0,NaN
3,7999,3454,0,0,3926002000,Sherwood Oaks,13605 NW County Road 235,Alachua,32615,Alachua,...,19.46,22.34,32.83,26.18,NaN,NaN,0,0,Construction Category Not Found,NaN
4,1356,0,800004435,0,3926020000,Sherwood Oaks Apartments,13400 NW 140th St,Alachua,32615,Alachua,...,19.46,22.34,32.83,26.18,48982.0,11.0,83b,2018-11-19,0,NaN


### df_assistance_program

Some rows have `0` in the Address and State columns instead of valid values. Since all entries in this dataset are located in Gainesville, Alachua County, we replace the `0` values in State with `FL` and mark missing addresses as `N/A`.

All columns that use `x` and `-` to indicate a yes/no relationship are converted to `True` and `False`.

In [ ]:
df_assistance_program["Address"] = df_assistance_program["Address"].replace(0, "N/A")
df_assistance_program["State"] = df_assistance_program["State"].replace(0, "FL")

funding_cols = df_assistance_program.columns[df_assistance_program.columns.get_loc("McKinney Vento-funded"):]
df_assistance_program[funding_cols] = df_assistance_program[funding_cols].replace({"-": False, "x": True})

df_assistance_program.head()

### df_lost_property

The `FHFC HPP key` and `HUD/RD Rental Assistance Units` columns use `-` for properties where the value doesn't apply. We replace these with `NaN` instead of `0` or a string like `"N/A"` so the columns stay numeric, missing values are automatically excluded from aggregations, and there's no confusion with a real value of zero.

The `Approximate Year of Loss` column lists `9999` as the year for Gardens of Gainesville, which we assume is a typo for `1999`.

The `Year Built/Funded`, `Type of Ownership`, and `Population Served` columns contain `not avail.` for properties where the information is unknown. We replace these with `N/A` for consistency.

All columns that use `x` and `-` to indicate a yes/no relationship are converted to `True` and `False`.

In [ ]:
import numpy as np

pd.set_option('display.max_columns', None)

df_lost_property["FHFC HPP key"] = df_lost_property["FHFC HPP key"].replace("-", np.nan)
df_lost_property["HUD/RD Rental Assistance Units"] = df_lost_property["HUD/RD Rental Assistance Units"].replace({"-": np.nan, " - ": np.nan})
df_lost_property["Approximate Year of Loss"] = df_lost_property["Approximate Year of Loss"].replace(9999, 1999)
df_lost_property["Year Built/Funded"] = df_lost_property["Year Built/Funded"].replace("not avail.", "N/A")
df_lost_property["Type of Ownership"] = df_lost_property["Type of Ownership"].replace("not avail.", "N/A")
df_lost_property["Population Served"] = df_lost_property["Population Served"].replace("not avail.", "N/A")

bool_cols = df_lost_property.columns[df_lost_property.columns.get_loc("FHFC Funded"):df_lost_property.columns.get_loc("RD Prepaid Mortgage") + 1]
df_lost_property[bool_cols] = df_lost_property[bool_cols].replace({"-": False, "x": True})

df_lost_property.head()

### df_housing_agencies

Gainesville Housing Authority has `0` for `Capital Fund`, which is likely a data entry issue.

> **Note:** We leave this value as-is since we don't have a reliable replacement, but it should be treated with caution in any analysis involving capital funding.

In [ ]:
df_housing_agencies.head()

### Export Cleaned Data

We export all 5 cleaned dataframes to CSV so they can be loaded directly in the EDA notebooks without repeating the cleaning steps.

In [11]:
df_funder.to_csv("../Data/Cleaned Data/df_funder.csv", index=False)
df_inventory.to_csv("../Data/Cleaned Data/df_inventory.csv", index=False)
df_assistance_program.to_csv("../Data/Cleaned Data/df_assistance_program.csv", index=False)
df_lost_property.to_csv("../Data/Cleaned Data/df_lost_property.csv", index=False)
df_housing_agencies.to_csv("../Data/Cleaned Data/df_housing_agencies.csv", index=False)